# ChaoSSM — Phase 18: classical RC benchmarks with *designed* memory depth

Addresses Reviewer 1 comments 2 and 4, and Reviewer 2 comment 6.

**Why these targets.** The two-factor account currently rests on three targets. NARMA10 and
NARMA30 are the standard reservoir-computing benchmarks *and* they have a memory depth that is
designed rather than merely measured: by construction the output depends on the previous 10 and
30 input steps. If the account is right, the critical arm's margin over the ordered arm should
grow from NARMA10 to NARMA30. That is a genuine out-of-sample prediction, not a fit. The Santa Fe
laser series adds a real measured system.

**Protocol.** Identical to Phase 14/16: recurrence *and* input encoder frozen and random, readout
only trained. Arms: ordered (λ≈−0.30), edge (λ≈0⁻), deep (λ≈+0.30), linear control, and a leaky
ESN (Reviewer 2 comment 6). Five seeds each.

**Two metrics.** NRMSE, standard in the NARMA literature, and the binned accuracy used elsewhere
in this paper, so all six targets sit on one scale in the demand-versus-margin figure.

Runtime: roughly 25–40 min on a T4. The run is resumable — rerun the cell after a disconnect.

## Setup

In [ ]:
import os, json, math, time, warnings, urllib.request
from dataclasses import dataclass, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))

def setup_results_dir(project="ChaoSSM",phase="phase18_rc_benchmarks"):
    """Prefer Google Drive so results survive the runtime; warn loudly if unavailable."""
    base=None
    try:
        from google.colab import drive
        try: drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}: {e1}); retrying force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78)
        print(f"[WARNING] Google Drive is NOT mounted ({type(e).__name__}).")
        print(f"[WARNING] Results go to {base} -- the runtime's LOCAL disk.")
        print("[WARNING] Those files are LOST when the runtime recycles. Run the rescue cell at")
        print("[WARNING] the end, or download them, as soon as the run finishes.")
        print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base

RESULTS=setup_results_dir()
# This notebook writes ONLY into results/phase18_rc_benchmarks/. No earlier phase is touched.
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=600,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)

# --- data directory on Drive (persists across runtime recycles) ---------------
DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(RESULTS)), "data")
os.makedirs(DATA_DIR, exist_ok=True)
SANTAFE_CSV = os.path.join(DATA_DIR, "santafe_laser.csv")
print("data dir ->", DATA_DIR)
print("Santa Fe expected at ->", SANTAFE_CSV)
print("  (place santafe_laser.csv there once; it then loads from Drive on every run)")


torch 2.11.0+cu130 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase18_rc_benchmarks
data dir -> /content/drive/My Drive/ChaoSSM/data
Santa Fe expected at -> /content/drive/My Drive/ChaoSSM/data/santafe_laser.csv
  (place santafe_laser.csv there once; it then loads from Drive on every run)


## Configuration

In [ ]:
SMOKE_TEST=False   # True runs a ~3 min code check; results meaningless

@dataclass
class Config:
    state_dim:int=64; d_model:int=64
    steps:int=2500; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; gamma_init:float=1.0; lr:float=1e-3
    seeds:int=5
    eval_n:int=1500; eval_seed:int=12345
    seq_len:int=64; fc_bins:int=32
    r_cap_map:float=4.0
    lambda_target:dict=field(default_factory=lambda: dict(ordered=-0.30, edge=-0.02, deep=0.30))
    lambda_tol:float=0.05; calib_steps:int=200; calib_warm:int=60
    chaos_regimes:tuple=("ordered","edge","deep")
    arms_extra:tuple=("linear-control","leaky-esn")
    targets:tuple=("narma10","narma30","santafe")
    train_frac:float=0.8          # temporal train/eval split (no window leakage)
    santafe_csv:str=""           # set in the next cell to the Drive path
    leak_rate:float=0.3; esn_rho:float=0.95

CFG=Config()
if SMOKE_TEST:
    CFG.steps=40; CFG.seeds=2; CFG.eval_n=60; CFG.calib_steps=30; CFG.calib_warm=10
    CFG.chaos_regimes=("ordered","edge"); CFG.targets=("narma10",)
    print("*** SMOKE TEST — results meaningless, this only checks the code runs ***")
n_runs=len(CFG.targets)*(len(CFG.chaos_regimes)+len(CFG.arms_extra))*CFG.seeds
print("targets:",CFG.targets)
print("arms:",CFG.chaos_regimes+CFG.arms_extra)
print("planned runs =",n_runs)

targets: ('narma10', 'narma30', 'santafe')
arms: ('ordered', 'edge', 'deep', 'linear-control', 'leaky-esn')
planned runs = 75


## Cells and γ-calibration — verbatim from Phase 14/15/16, plus a leaky-ESN arm

In [ ]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaosCell(nn.Module):
    def __init__(self,cfg,regime):
        super().__init__(); self.N=cfg.state_dim; self.regime=regime
        self.r_cap=float(cfg.r_cap_map)
        self.w_r=nn.Parameter(torch.zeros(self.N))
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad(): self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.w_r,self.W_mix,self.w_gamma]: p_.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LinearCell(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W_lin.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LeakyESNCell(nn.Module):
    """Standard leaky-integrator ESN (Reviewer 2, comment 6): spectral radius rho, leak rate a."""
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim; self.a=float(cfg.leak_rate)
        W=torch.randn(self.N,self.N)/math.sqrt(self.N)
        ev=torch.linalg.eigvals(W).abs().max().item()
        self.W=nn.Parameter(W*(cfg.esn_rho/max(ev,1e-9)))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def step(self,h,x): return (1-self.a)*h + self.a*torch.tanh(h@self.W.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0,steps=None,warm=None):
    steps=steps or cfg.calib_steps; warm=warm or cfg.calib_warm; N=cfg.state_dim
    dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0.0,1.0)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0):
    target=cfg.lambda_target[regime]; lo,hi=0.01,8.0; best=None; best_err=1e9
    for _ in range(18):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1],bool(best_err<cfg.lambda_tol)

def make_cell(cfg,regime):
    if regime=="linear-control": return LinearCell(cfg)
    if regime=="leaky-esn":      return LeakyESNCell(cfg)
    return ChaosCell(cfg,regime)

## Targets

**NARMA10 / NARMA30** are *driven* systems: the network receives the input `u(t)` and must output
`y(t)`, where `y` depends on the previous 10 (resp. 30) steps of `u` and `y` by construction. This
is the standard benchmark form, and it is what makes the memory depth designed rather than merely
observed.

**Santa Fe laser** is the classical one-step-ahead prediction task on real measured data. If the
download fails the cell says so and the target is skipped — the rest of the phase still runs.

In [ ]:
def narma(n, order=10, seed=0, max_tries=40):
    """Standard NARMA-m. Returns (u, y). Regenerates on divergence, which NARMA30 can show."""
    rng=np.random.default_rng(seed)
    if order==10: a,b,c,d = 0.3, 0.05, 1.5, 0.1
    else:         a,b,c,d = 0.2, 0.004, 1.5, 0.001
    for attempt in range(max_tries):
        u=rng.uniform(0,0.5,n+order+200)
        y=np.zeros_like(u)
        ok=True
        for t in range(order, len(u)-1):
            y[t+1]=a*y[t] + b*y[t]*np.sum(y[t-order+1:t+1]) + c*u[t-order+1]*u[t] + d
            if not np.isfinite(y[t+1]) or abs(y[t+1])>1e3: ok=False; break
        if ok: return u[order+200:], y[order+200:]
        rng=np.random.default_rng(seed+1000*(attempt+1))
    raise RuntimeError(f"NARMA{order} diverged on every attempt")

_SANTAFE=None
def santafe_series():
    """Load the Santa Fe laser series (set A) from the CSV on Drive. Native length, no tiling.
    Provenance: Weigend & Gershenfeld (1993), Santa Fe Time Series Prediction Competition;
    one integer intensity per line, 0-255. Put santafe_laser.csv in DATA_DIR once."""
    global _SANTAFE
    if _SANTAFE is None:
        path = CFG.santafe_csv or SANTAFE_CSV
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Santa Fe CSV not found at {path}. Upload santafe_laser.csv to that folder on "
                f"your Drive (the file is provided alongside this notebook), then re-run.")
        _SANTAFE = np.loadtxt(path, comments="#").astype(float)
        print(f"  santafe: {len(_SANTAFE)} points loaded from {path} "
              f"(range {_SANTAFE.min():.0f}-{_SANTAFE.max():.0f}, "
              f"{int((np.diff(_SANTAFE)<-100).sum())} collapse events)")
    return _SANTAFE

def normalize01(x):
    lo,hi=np.min(x),np.max(x); return (x-lo)/(hi-lo+1e-9)

_SRC={}
def get_target(name, seed, length=60000):
    """Returns dict(mode, inp, out, depth, n_train). Series carry a temporal split index
    n_train: training windows are drawn from [0, n_train), eval windows from [n_train, n);
    the two never share underlying samples, so there is no window leakage."""
    key=(name,seed)
    if key not in _SRC:
        if name.startswith("narma"):
            order=int(name.replace("narma",""))
            u,y=narma(length, order=order, seed=seed)   # fresh 60k series per seed
            inp,out=normalize01(u),normalize01(y); depth=order
        elif name=="santafe":
            x=normalize01(santafe_series())             # native 10093 points, NO tiling
            inp,out=x,x; depth=None
        else:
            raise ValueError(name)
        n=len(inp); n_train=int(CFG.train_frac*n)
        _SRC[key]=dict(mode=("driven" if name.startswith("narma") else "auto"),
                       inp=inp, out=out, depth=depth, n_train=n_train)
    return _SRC[key]

def make_batch(cfg,batch,device,gen,name,src_seed,split="train"):
    """Draw `batch` windows from the train portion [0,n_train) or the eval portion
    [n_train,n). The two portions are disjoint, so train and eval never share samples."""
    S=get_target(name,src_seed); L=cfg.seq_len; nt=S["n_train"]; n=len(S["inp"])
    lo,hi=(0,nt-L-1) if split=="train" else (nt,n-L-1)
    g=np.random.default_rng(int(torch.randint(0,2**31-1,(1,),generator=gen,device=gen.device).item()))
    starts=g.integers(lo,hi,size=batch)
    if S["mode"]=="driven":
        U=np.stack([S["inp"][s:s+L] for s in starts]); Y=np.stack([S["out"][s:s+L] for s in starts])
        inp=torch.tensor(U,dtype=torch.float32,device=device); ytrue=torch.tensor(Y,dtype=torch.float32,device=device)
    else:
        X=np.stack([S["inp"][s:s+L+1] for s in starts])
        inp=torch.tensor(X[:,:L],dtype=torch.float32,device=device)
        ytrue=torch.tensor(X[:,1:L+1],dtype=torch.float32,device=device)
    tgt=torch.clamp((ytrue*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp,tgt,ytrue

def majority_chance(cfg,name,src_seed):
    S=get_target(name,src_seed)
    y=S["out"] if S["mode"]=="driven" else S["inp"][1:]
    b=np.clip((y*cfg.fc_bins).astype(int),0,cfg.fc_bins-1)
    cnt=np.bincount(b,minlength=cfg.fc_bins); return float(cnt.max())/float(cnt.sum())

for nm in CFG.targets:
    try:
        S=get_target(nm,0,8000)
        print(f"{nm:10s} mode={S['mode']:7s} depth={S['depth']} "
              f"chance={majority_chance(CFG,nm,0):.4f}")
    except Exception as e:
        print(f"{nm:10s} UNAVAILABLE: {e}")

narma10    mode=driven  depth=10 chance=0.1064
narma30    mode=driven  depth=30 chance=0.0921
  santafe: 10093 points loaded from /content/drive/My Drive/ChaoSSM/data/santafe_laser.csv (range 0-255, 232 collapse events)
santafe    mode=auto    depth=None chance=0.1209


## Readout model — canonical: recurrence *and* input encoder frozen, readout only

In [ ]:
class ReservoirLM(nn.Module):
    def __init__(self,cfg,cell,out_dim):
        super().__init__(); self.inp=nn.Linear(1,cfg.d_model); self.cell=cell
        self.ln=nn.LayerNorm(cfg.state_dim); self.readout=nn.Linear(cfg.state_dim,out_dim)
        for p_ in self.inp.parameters(): p_.requires_grad_(False)   # canonical: random, frozen drive
    def states(self,x):
        with torch.no_grad():
            e=self.inp(x.unsqueeze(-1)); B,T=e.shape[0],e.shape[1]
            h=self.cell.init_state(B,e.device,e.dtype); H=[]
            for t in range(T):
                h=self.cell.step(h,e[:,t]); H.append(h)
            return torch.stack(H,1)
    def forward(self,x): return self.readout(self.ln(self.states(x)))

def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

@torch.no_grad()
def evaluate(model,cfg,name,src_seed,chunk=256):
    """Returns (binned accuracy, NRMSE). Accuracy keeps this comparable with the other targets;
    NRMSE is the metric the NARMA literature reports."""
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    inp,tgt,ytrue=make_batch(cfg,cfg.eval_n,DEVICE,g,name,src_seed,split="eval")
    model.eval(); corr=tot=0.0; se=[]; yy=[]
    centers=(torch.arange(cfg.fc_bins,device=DEVICE,dtype=torch.float32)+0.5)/cfg.fc_bins
    for i in range(0,inp.shape[0],chunk):
        lg=model(inp[i:i+chunk]); p=lg.argmax(-1); t=tgt[i:i+chunk]
        corr+=(p==t).float().sum().item(); tot+=t.numel()
        yhat=centers[p]                      # bin centre as the point estimate
        se.append(((yhat-ytrue[i:i+chunk])**2).reshape(-1)); yy.append(ytrue[i:i+chunk].reshape(-1))
    model.train()
    se=torch.cat(se); yy=torch.cat(yy)
    nrmse=float(torch.sqrt(se.mean()/ (yy.var()+1e-12)))
    return corr/max(tot,1), nrmse

## Run — resumable, checkpoints after every run

In [ ]:
FRESH_START=False
CKPT=os.path.join(RESULTS,"phase18_rc_benchmarks.csv")
COLS=["target","regime","seed","diverged","acc","nrmse","calibrated_lambda","calibration_ok","n_trainable"]

if FRESH_START and os.path.exists(CKPT):
    os.replace(CKPT,CKPT+".bak"); print("[resume] FRESH_START -> previous CSV moved to",CKPT+".bak")
if os.path.exists(CKPT):
    rows=pd.read_csv(CKPT).to_dict("records")
    done={(r["target"],r["regime"],int(r["seed"])) for r in rows}
    print(f"[resume] found {len(rows)} completed runs")
else:
    rows=[]; done=set(); print("[resume] no checkpoint — starting from scratch")

def _flush(): pd.DataFrame(rows)[COLS].to_csv(CKPT,index=False)

ARMS=list(CFG.chaos_regimes)+list(CFG.arms_extra)
avail=[]
for t in CFG.targets:
    try: get_target(t,0,8000); avail.append(t)
    except Exception as e: print(f"[skip] {t}: {e}")
planned=[(t,r,s) for t in avail for r in ARMS for s in range(CFG.seeds)]
todo=[k for k in planned if k not in done]
print(f"[resume] {len(planned)} planned, {len(planned)-len(todo)} done, {len(todo)} to run\n")

t0=time.time()
for target,regime,sd in todo:
    torch.manual_seed(sd); np.random.seed(sd)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(sd)
    lam=float("nan"); ok=True
    cell=make_cell(CFG,regime).to(DEVICE)
    if regime in CFG.chaos_regimes:
        _,lam,ok=calibrate_gamma(cell,CFG,regime,seed=sd)
        if not ok: print(f"    [calibration WARNING] {target}/{regime} s={sd}: lambda={lam:+.3f} missed target")
    cell.freeze_recurrence(); cell.freeze_input()
    model=ReservoirLM(CFG,cell,CFG.fc_bins).to(DEVICE)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=CFG.lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(CFG,CFG.steps))
    div=False
    for step in range(CFG.steps):
        inp,tgt,_=make_batch(CFG,CFG.batch,DEVICE,gen,target,sd,split="train")
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1))
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(params,CFG.grad_clip).item(); opt.step(); sched.step()
        if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
    if div: acc,nrm=float("nan"),float("nan")
    else:   acc,nrm=evaluate(model,CFG,target,sd)
    rows.append(dict(target=target,regime=regime,seed=sd,diverged=div,acc=acc,nrmse=nrm,
                     calibrated_lambda=lam,calibration_ok=ok,
                     n_trainable=sum(p.numel() for p in params)))
    _flush()
    lt="" if math.isnan(lam) else f" lam={lam:+.3f}"
    print(f"[P18] {target:9s} {regime:14s} s={sd} acc={acc:.4f} nrmse={nrm:.4f}{lt}"+(" DIVERGED" if div else ""))

RES=pd.DataFrame(rows)[COLS]; _flush()
print(f"\nwall clock this session: {(time.time()-t0)/60:.1f} min | total runs: {len(RES)}")
print("n_trainable:",sorted(RES.n_trainable.unique()),"(all arms must match — readout only)")

[resume] found 28 completed runs
[resume] 75 planned, 28 done, 47 to run

[P18] narma30   ordered        s=3 acc=0.1010 nrmse=0.9242 lam=-0.276
[P18] narma30   ordered        s=4 acc=0.1089 nrmse=0.9303 lam=-0.277
[P18] narma30   edge           s=0 acc=0.1247 nrmse=0.9513 lam=+0.015
[P18] narma30   edge           s=1 acc=0.1135 nrmse=0.9622 lam=+0.009
[P18] narma30   edge           s=2 acc=0.1059 nrmse=0.9808 lam=+0.009
[P18] narma30   edge           s=3 acc=0.1064 nrmse=0.9610 lam=+0.017
[P18] narma30   edge           s=4 acc=0.1082 nrmse=0.9681 lam=+0.013
[P18] narma30   deep           s=0 acc=0.1041 nrmse=1.0917 lam=+0.272
[P18] narma30   deep           s=1 acc=0.1052 nrmse=1.0582 lam=+0.293
[P18] narma30   deep           s=2 acc=0.1014 nrmse=1.0807 lam=+0.321
[P18] narma30   deep           s=3 acc=0.0996 nrmse=1.0944 lam=+0.306
[P18] narma30   deep           s=4 acc=0.1052 nrmse=1.0285 lam=+0.273
[P18] narma30   linear-control s=0 acc=0.1171 nrmse=0.9358
[P18] narma30   linear-cont

## Memory demand of each new target

Same estimator as Phase 17, so the six targets are directly comparable: lagged mutual information
between the driving input at lag *k* and the quantity the network must output. For NARMA this is
`MI(u(t−k); y(t))`, and it should fall off at the designed order. For Santa Fe it is
`MI(x(t−k); x(t+1))`.

In [ ]:
def mutual_info(a,b,bins=24):
    """Histogram MI in nats. Same estimator and bin count as Phase 17."""
    H,_,_=np.histogram2d(a,b,bins=bins)
    P=H/max(H.sum(),1); Pa=P.sum(1,keepdims=True); Pb=P.sum(0,keepdims=True)
    nz=P>0
    return float(np.sum(P[nz]*np.log(P[nz]/(Pa@Pb)[nz])))

LAGS=list(range(1,41))
dem_rows=[]
for t in avail:
    S=get_target(t,0,40000)
    drive = S["inp"]; out = S["out"] if S["mode"]=="driven" else S["inp"]
    for k in LAGS:
        if S["mode"]=="driven":
            a=drive[:-k]; b=out[k:]                       # MI(u(t-k); y(t))
        else:
            a=drive[:-(k+1)]; b=out[k+1:]                 # MI(x(t-k); x(t+1))
        m=min(len(a),len(b),30000)
        dem_rows.append(dict(series=t,lag=k,mutual_info=mutual_info(a[:m],b[:m])))
DEM=pd.DataFrame(dem_rows); savetable(DEM,"phase18_series_memory_demand.csv")

print("long-lag demand (mean MI at lags >= 10), the x-axis of the demand-margin figure:")
for t in avail:
    d=DEM[(DEM.series==t)&(DEM.lag>=10)].mutual_info.mean()
    print(f"  {t:10s} {d:.4f} nats")
print("\nfor reference, the three targets already in the paper:")
print("  logistic   0.0651 | nonchaotic 0.2088 | mackey_glass 0.6220")

saved /content/drive/My Drive/ChaoSSM/results/phase18_rc_benchmarks/phase18_series_memory_demand.csv
long-lag demand (mean MI at lags >= 10), the x-axis of the demand-margin figure:
  narma10    0.0382 nats
  narma30    0.0461 nats
  santafe    0.3702 nats

for reference, the three targets already in the paper:
  logistic   0.0651 | nonchaotic 0.2088 | mackey_glass 0.6220


## Analysis

In [ ]:
def ci(x,seed=0,n=20000):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)==0: return (float("nan"),)*2
    rng=np.random.default_rng(seed); b=rng.choice(x,(n,len(x))).mean(1)
    return float(np.percentile(b,2.5)), float(np.percentile(b,97.5))

summ=[]
for t in avail:
    d=RES[RES.target==t]
    o=d[d.regime=="ordered"].sort_values("seed").acc.values
    e=d[d.regime=="edge"].sort_values("seed").acc.values
    p=e-o; lo,hi=ci(p,1)
    row=dict(target=t,
             ordered=round(float(np.nanmean(o)),4), edge=round(float(np.nanmean(e)),4),
             deep=round(float(np.nanmean(d[d.regime=="deep"].acc)),4),
             linear=round(float(np.nanmean(d[d.regime=="linear-control"].acc)),4),
             leaky_esn=round(float(np.nanmean(d[d.regime=="leaky-esn"].acc)),4),
             edge_minus_ordered=round(float(np.nanmean(p)),4),
             ci=f"[{lo:+.4f},{hi:+.4f}]", edge_wins=int(np.nansum(p>0)), n=int(np.sum(np.isfinite(p))),
             demand=round(float(DEM[(DEM.series==t)&(DEM.lag>=10)].mutual_info.mean()),4))
    summ.append(row)
SUMM=pd.DataFrame(summ); savetable(SUMM,"phase18_summary.csv")
print(SUMM.to_string(index=False))
print("\nNRMSE (standard NARMA metric), mean over seeds:")
print(RES.groupby(["target","regime"]).nrmse.mean().round(4).to_string())
print("\nTHE PREDICTION UNDER TEST: edge-minus-ordered should be larger for narma30 than narma10,")
print("because NARMA30's designed memory depth is three times deeper.")

saved /content/drive/My Drive/ChaoSSM/results/phase18_rc_benchmarks/phase18_summary.csv
 target  ordered   edge   deep  linear  leaky_esn  edge_minus_ordered                ci  edge_wins  n  demand
narma10   0.1352 0.1378 0.1325  0.1394     0.1440              0.0026 [+0.0011,+0.0040]          5  5  0.0382
narma30   0.1113 0.1118 0.1031  0.1106     0.1002              0.0004 [-0.0019,+0.0030]          2  5  0.0461
santafe   0.3306 0.3588 0.3005  0.2053     0.1157              0.0282 [+0.0176,+0.0389]          5  5  0.3702

NRMSE (standard NARMA metric), mean over seeds:
target   regime        
narma10  deep              0.9624
         edge              0.8979
         leaky-esn         0.8949
         linear-control    0.8831
         ordered           0.9041
narma30  deep              1.0707
         edge              0.9647
         leaky-esn         1.1906
         linear-control    0.9343
         ordered           0.9257
santafe  deep              0.6094
         edge            

## Figures (600 dpi, MDPI requirement)

In [ ]:
C={"ordered":"#4C72B0","edge":"#2E7D32","deep":"#B71C1C",
   "linear-control":"#9E9E9E","leaky-esn":"#E8900C"}
LBL={"ordered":"ordered\n$\\lambda\\approx-0.30$","edge":"edge\n$\\lambda\\approx0^-$",
     "deep":"deep\n$\\lambda\\approx+0.30$","linear-control":"linear\nreservoir",
     "leaky-esn":"leaky\nESN"}
arms=list(CFG.chaos_regimes)+list(CFG.arms_extra)

fig,axes=plt.subplots(1,len(avail),figsize=(5.2*len(avail),4.6),squeeze=False)
for ax,t in zip(axes[0],avail):
    d=RES[RES.target==t]
    m=[np.nanmean(d[d.regime==a].acc) for a in arms]
    s=[np.nanstd(d[d.regime==a].acc,ddof=1) for a in arms]
    ax.bar(range(len(arms)),m,yerr=s,capsize=4,color=[C[a] for a in arms],width=.72,
           error_kw=dict(elinewidth=1.3,capthick=1.3))
    ax.axhline(np.nanmean(d[d.regime=="linear-control"].acc),ls="--",color="#777",lw=1.2)
    ax.set_xticks(range(len(arms))); ax.set_xticklabels([LBL[a] for a in arms],fontsize=8.5)
    ax.set_title(t,fontsize=12); ax.grid(axis="y",alpha=.25); ax.set_axisbelow(True)
axes[0][0].set_ylabel("Accuracy (frozen reservoir,\ncanonical protocol)")
fig.suptitle("Classical RC benchmarks under the canonical protocol (mean $\\pm$ sd, $n=5$)",fontsize=12.5)
fig.tight_layout(); savefig(fig,"phase18_benchmarks.png")

# demand vs margin, all six targets
PRIOR={"logistic":(0.0651,-0.0108),"nonchaotic_control":(0.2088,0.0085),"mackey_glass":(0.6220,0.0612)}
fig,ax=plt.subplots(figsize=(7.2,5.0))
for nm,(dx,dy) in PRIOR.items():
    ax.plot(dx,dy,"o",ms=9,color="#777"); ax.annotate(nm,(dx,dy),xytext=(6,-12),
            textcoords="offset points",fontsize=8.5,color="#555")
for r in summ:
    ax.plot(r["demand"],r["edge_minus_ordered"],"o",ms=10,color="#2E7D32")
    ax.annotate(r["target"],(r["demand"],r["edge_minus_ordered"]),xytext=(6,8),
                textcoords="offset points",fontsize=9,color="#2E7D32",fontweight="bold")
ax.axhline(0,ls="--",color="#777",lw=1.2)
ax.set_xlabel("task memory demand: mean MI at lags $\\geq 10$ (nats)")
ax.set_ylabel("edge $-$ ordered accuracy")
ax.set_title("Does the margin still track demand with the new benchmarks?",fontsize=12)
ax.grid(alpha=.25)
fig.tight_layout(); savefig(fig,"phase18_demand_vs_margin.png")
print("\ngrey = the three targets already in the paper; green = the three added here.")

saved /content/drive/My Drive/ChaoSSM/results/phase18_rc_benchmarks/phase18_benchmarks.png
saved /content/drive/My Drive/ChaoSSM/results/phase18_rc_benchmarks/phase18_demand_vs_margin.png

grey = the three targets already in the paper; green = the three added here.


## Rescue — run this if Drive was not mounted

In [ ]:
import shutil
if not RESULTS.startswith("/content/drive"):
    z=shutil.make_archive("/content/phase18_results","zip",RESULTS)
    print("Drive was NOT mounted. Archive written to:",z)
    try:
        from google.colab import files; files.download(z)
    except Exception as e:
        print("auto-download unavailable:",e,"-- download",z,"from the file browser.")
else:
    print("Results are on Drive:",RESULTS)

Results are on Drive: /content/drive/My Drive/ChaoSSM/results/phase18_rc_benchmarks
